# WP04 — اجرای Tripoli با حساب شما
این نوت‌بوک ورودی‌های اعلام‌شدهٔ حامد را استفاده می‌کند. کد مرجع GEE و موتور مدل تغییر نمی‌کنند.

**از یک Runtime تازه استفاده کن و سلول‌ها را به ترتیب اجرا کن.** ورود Google فقط در سلول ۲ انجام می‌شود؛ رمز یا توکن را در GitHub یا چت قرار نده.

خروجی، نتیجهٔ JSON، گراف‌های نقشه و شواهد اجرای Python است؛ فایل GeoTIFF تولید نمی‌شود.
مقایسه با GEE پس از اجرای جداگانهٔ مرجع انجام می‌شود؛ موفقیت Python به‌تنهایی تأیید تطابق نیست.

پروژه: `practical-proxy-441422-n6`  
محدوده: `projects/practical-proxy-441422-n6/assets/Libya/Tripoli`  
آموزش: `projects/practical-proxy-441422-n6/assets/Libya/TD3-attept1-tripoli`

تنظیمات فعلی حفظ شده‌اند: فیلد `lc_code`، سال ۲۰۲۴، بازهٔ پنج‌ساله، fallback سال ۲۰۲۳، ۵۰۰ درخت و سهم آموزش ۰٫۷.
دسترسی و موجود بودن این Assetها هنوز در این نوت‌بوک تأیید نشده‌اند.

## ۱. نصب نسخهٔ ثابت موتور
این نوت‌بوک عمداً commit آزموده‌شدهٔ WP04 را نصب می‌کند. شناسهٔ منبع اجرای واقعی همین commit خواهد بود.

In [ ]:
from pathlib import Path
import subprocess, sys, json, hashlib

ENGINE_COMMIT = "bf9cf6c8bbd03d083c2d9bccae722d7036923afc"
REPO_URL = "https://github.com/hamedsabzchi/libya-automated-land-cover-mapping.git"
CHECKOUT = Path("/content/libya-landcover-wp04")
if not CHECKOUT.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(CHECKOUT)], check=True)
    subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--quiet", "--detach", ENGINE_COMMIT], check=True)
actual = subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True).strip()
if actual != ENGINE_COMMIT:
    raise RuntimeError("Existing checkout has another version. Start a fresh Colab runtime.")
subprocess.run(["git", "-C", str(CHECKOUT), "diff", "--exit-code", "HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", str(CHECKOUT)], check=True)

import ee
from libya_landcover import cli, request_from_inputs
from libya_landcover.bundle import write_json
from libya_landcover.evidence import validate_probes
from contracts.models import GEE_SHA256
from python import landcover_engine, assistant_adapter
from contracts import models
if ee.__version__ != "1.7.48":
    raise RuntimeError("EE version differs. Restart the runtime and rerun from cell 1.")
for module, path in [(landcover_engine, "python/landcover_engine.py"),
                     (assistant_adapter, "python/assistant_adapter.py"),
                     (models, "contracts/models.py")]:
    if Path(module.__file__).read_bytes() != (CHECKOUT/path).read_bytes():
        raise RuntimeError("An older module is already loaded. Start a fresh runtime.")
reference = CHECKOUT / "gee/automated_land_cover_app.js"
assert hashlib.sha256(reference.read_bytes()).hexdigest() == GEE_SHA256
print("Pinned engine installed; GEE reference checksum verified:", ENGINE_COMMIT)

## ۲. ورود به Earth Engine
با حسابی وارد شو که به پروژه و هر دو Asset دسترسی دارد. اگر ورود یا دسترسی خطا داد، اجرای سلول‌های بعد را متوقف کن.

In [ ]:
PROJECT = "practical-proxy-441422-n6"
ee.Authenticate()
ee.Initialize(project=PROJECT)
print("Earth Engine initialized for", PROJECT)
print("Asset access will be checked next; initialization alone does not verify it.")

## ۳. آماده‌سازی درخواست و نقاط مقایسه
این سلول فقط محدودهٔ جغرافیایی و مرکز Asset را می‌خواند. ۲۵ نقطه روی گسترهٔ محدوده و یک نقطه در مرکز هندسی ساخته می‌شود؛ برخی نقاط ممکن است خارج از پلیگون باشند تا ماسک نیز مشاهده شود.
همان فایل نقاط برای هر دو اجرای Python و GEE استفاده خواهد شد. این نقاط، دادهٔ آموزش یا روش ارزیابی مدل را تغییر نمی‌دهند.

In [ ]:
from uuid import uuid4
from datetime import datetime, timezone
import math

CASE_ROOT = Path("/content") / ("wp04-tripoli-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8])
INPUT = CASE_ROOT / "input"
INPUT.mkdir(parents=True, exist_ok=False)
request = request_from_inputs({
    "request_id": str(uuid4()),
    "aoi_asset": "projects/practical-proxy-441422-n6/assets/Libya/Tripoli",
    "training_asset": "projects/practical-proxy-441422-n6/assets/Libya/TD3-attept1-tripoli",
    "class_field": "lc_code", "target_year": 2024, "year_count": 5,
    "fallback_year": 2023, "train_fraction": 0.7, "trees": 500,
})
write_json(INPUT / "request.json", request.to_dict())
write_json(INPUT / "engine-settings.json", request.engine_config())
write_json(INPUT / "context.json", {"project": PROJECT, "engine_commit": ENGINE_COMMIT})
roi = ee.FeatureCollection(request.to_dict()["aoi_asset"])
geometry = roi.geometry()
extent = ee.Dictionary({
    "feature_count": roi.size(),
    "bounds": geometry.bounds(1, ee.Projection("EPSG:4326")).coordinates(),
    "centroid": geometry.centroid(1, ee.Projection("EPSG:4326")).coordinates(),
}).getInfo()
if extent["feature_count"] < 1:
    raise RuntimeError("The supplied region asset is empty.")
ring = extent["bounds"][0]
west, east = min(p[0] for p in ring), max(p[0] for p in ring)
south, north = min(p[1] for p in ring), max(p[1] for p in ring)
lon, lat = extent["centroid"]
if not (west < east and south < north and east-west < 180 and -80 <= lat <= 84):
    raise RuntimeError("This extent needs an explicit probe-grid choice; no model run started.")
zone = min(60, max(1, math.floor((lon+180)/6)+1))
crs = "EPSG:" + str((32600 if lat >= 0 else 32700) + zone)
points = [{"id": f"grid-{row}-{col}", "lon": west+(east-west)*col/4,
           "lat": south+(north-south)*row/4} for row in range(5) for col in range(5)]
points.append({"id": "centroid", "lon": lon, "lat": lat})
probes = validate_probes({"crs": crs, "transform": [10,0,0,0,-10,0], "points": points})
write_json(INPUT / "probes.json", probes)
write_json(INPUT / "region-observation.json", extent)
print("Output folder:", CASE_ROOT)
print("Reference settings (use the same settings in GEE):")
print(json.dumps(request.engine_config(), indent=2))
print("Probe locations:", len(points), "CRS:", crs, "Grid:", probes["transform"])

## ۴. اجرای موتور و ثبت شواهد
این سلول اجرای واقعی Earth Engine را آغاز می‌کند. تا پایان آن صبر کن؛ زمان اجرا به داده‌ها و سرویس بستگی دارد.
اگر اجرای مدل یا دریافت شواهد خطا داد، وضعیت ثبت می‌شود. بدون تغییر کد مرجع، فایل تشخیص GEE نیز در پوشهٔ خروجی ساخته خواهد شد.

In [ ]:
SESSION = CASE_ROOT / "reference-session.js"
if not SESSION.exists():
    assert cli.main(["prepare-reference", "--reference", str(reference),
                     "--probes", str(INPUT/"probes.json"), "--output", str(SESSION)]) == 0
RUN = CASE_ROOT / "run"
if RUN.exists():
    raise RuntimeError("This run folder already exists. Download it first; rerun cell 3 for a new attempt.")
exit_code = cli.main(["run", str(INPUT/"request.json"), "--project", PROJECT,
                     "--engine-commit", ENGINE_COMMIT, "--output", str(RUN),
                     "--probes", str(INPUT/"probes.json")])
print("Runner exit code:", exit_code)
if (RUN/"manifest.json").exists():
    print((RUN/"manifest.json").read_text())
if (RUN/"status.json").exists():
    status = json.loads((RUN/"status.json").read_text())
    if status.get("error"):
        print("Engine error:", json.dumps(status["error"], ensure_ascii=False))
if exit_code == 0:
    print("Python run complete. GEE comparison has NOT yet been performed.")
else:
    print("Run or evidence collection did not complete. Download the bundle in cell 5 for inspection.")

## ۵. دانلود بستهٔ نتیجه
فایل ZIP را دانلود کن و برای ادامهٔ بررسی در چت بفرست. حتی اگر سلول ۴ خطا داد، این سلول پوشهٔ موجود و گزارش‌های ثبت‌شده را جمع‌آوری می‌کند.
این بسته فقط پوشهٔ اجرای حاضر را شامل می‌شود؛ فایل‌های ورود یا کل پوشهٔ Colab جمع‌آوری نمی‌شوند.

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(str(CASE_ROOT), "zip", root_dir=CASE_ROOT)
files.download(archive)
print("Downloaded:", Path(archive).name)

## ۶. مرحلهٔ اختیاری بعدی: خروجی مرجع GEE و مقایسه
پس از موفقیت اجرای Python:

1. فایل `reference-session.js` داخل ZIP را در یک اسکریپت موقت Code Editor باز کن. فایل اصلی مرجع در GitHub تغییر نمی‌کند.
2. پروژه و دو مسیر Asset و تمام تنظیمات نمایش‌داده‌شده در سلول ۳ را وارد کن؛ دکمهٔ RUN اصلی را بزن.
3. پس از پایان، دکمهٔ **WP04: capture execution observations** را بزن.
4. رشتهٔ JSON کامل چاپ‌شده را بدون برچسب کنسول و بدون کوتیشن بیرونی در `gee-evidence.json` ذخیره کن.
5. در سلول زیر مقدار `RUN_GEE_COMPARISON` را `True` کن، همان سلول را اجرا و فایل JSON را بارگذاری کن. سپس سلول ۵ را دوباره برای دانلود بستهٔ به‌روز اجرا کن.

`observations_match` فقط برابری همین مشاهدات را نشان می‌دهد؛ اثبات برابری تمام پیکسل‌های نقشه نیست.

In [ ]:
RUN_GEE_COMPARISON = False
if RUN_GEE_COMPARISON:
    if not (RUN/"python-evidence.json").exists():
        raise RuntimeError("Python evidence is missing. Resolve that run before comparison.")
    uploaded = files.upload()
    if len(uploaded) != 1 or "gee-evidence.json" not in uploaded:
        raise ValueError("Upload exactly one file named gee-evidence.json.")
    target = RUN/"gee-evidence.json"
    with target.open("xb") as stream:
        stream.write(uploaded["gee-evidence.json"])
    from libya_landcover.evidence import compare
    report = compare(cli.read_json(RUN/"python-evidence.json"), cli.read_json(target))
    write_json(RUN/"comparison.json", report)
    print(json.dumps(report, indent=2))
    print("Rerun cell 5 to download the updated bundle.")
else:
    print("GEE comparison not requested yet. No upload dialog or comparison was started.")